# Colab: Top‑10 Subreddit Frequency by Stress Label
Reads JSONL from Drive, finds top‑10 subreddits for **stress (label=1)** and **non‑stress (label=0)**, saves CSVs and shows charts.

In [1]:

# 0) Setup
!pip -q install --upgrade pandas pyarrow
import os, json
import pandas as pd
from google.colab import drive

# Mount Drive
drive.mount("/content/drive")

# Paths provided
base_drive_dir = "/content/drive/My Drive/Rice University/25fall/ELEC509/Final Project/Dataset/Stress Analysis in Social Media"
os.makedirs(base_drive_dir, exist_ok=True)

train_out = os.path.join(base_drive_dir, "dreaddit_train.jsonl")
test_out  = os.path.join(base_drive_dir, "dreaddit_test.jsonl")

print("Train JSONL:", train_out)
print("Test  JSONL:", test_out)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
Mounted at /content

In [2]:

# 1) Load JSONL files
import pandas as pd, json

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                print("Skipping line:", e)
    return pd.DataFrame(rows)

df_train = load_jsonl(train_out)
df_test  = load_jsonl(test_out)
print("Loaded:", len(df_train), "train,", len(df_test), "test")

df = pd.concat([df_train, df_test], ignore_index=True)
print("Total rows:", len(df))

# Normalize and coerce types
df.rename(columns={c: c.lower() for c in df.columns}, inplace=True)
for col in ["label", "subreddit"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")
df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
df["subreddit"] = df["subreddit"].astype(str)

df.head(3)


Loaded: 2838 train, 715 test
Total rows: 3553


,input_text,output_text,prediction,label,id,subreddit,text
0,User ID: 33181\nQuery: User ID: 33181 asked in...,"yes, the user feels stressful right now",1,1,33181,ptsd,"he said he had not felt that way before, sugge..."
1,User ID: 2606\nQuery: User ID: 2606 asked in t...,"no, the user does not feel stressful right now",0,0,2606,assistance,"hey there r/assistance, not sure if this is th..."
2,User ID: 38816\nQuery: User ID: 38816 asked in...,"yes, the user feels stressful right now",1,1,38816,ptsd,my mom then hit me with the newspaper and it s...


In [3]:

# 2) Compute top-10 subreddits by label
top_k = 10

stress_counts = (
    df[df["label"] == 1]
    .groupby("subreddit", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(top_k)
    .reset_index(drop=True)
)

nonstress_counts = (
    df[df["label"] == 0]
    .groupby("subreddit", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(top_k)
    .reset_index(drop=True)
)

print("Top-10 stress-related subreddits:")
display(stress_counts)
print("Top-10 non-stress subreddits:")
display(nonstress_counts)


Top-10 stress-related subreddits:


,subreddit,size
0,anxiety,416
1,ptsd,414
2,relationships,307
3,domesticviolence,249
4,survivorsofabuse,143
5,assistance,126
6,homeless,81
7,almosthomeless,59
8,stress,45
9,food_pantry,17


Top-10 non-stress subreddits:


,subreddit,size
0,relationships,387
1,ptsd,297
2,anxiety,234
3,assistance,229
4,survivorsofabuse,172
5,domesticviolence,139
6,homeless,139
7,almosthomeless,40
8,stress,33
9,food_pantry,26


In [ ]:

# 3) Save CSV summaries back to Drive
stress_csv_path = os.path.join(base_drive_dir, "top10_stress_subreddits.csv")
nonstress_csv_path = os.path.join(base_drive_dir, "top10_nonstress_subreddits.csv")

stress_counts.to_csv(stress_csv_path, index=False)
nonstress_counts.to_csv(nonstress_csv_path, index=False)

print("Saved:")
print(" ", stress_csv_path)
print(" ", nonstress_csv_path)


In [ ]:

# 4) Plot bar charts (matplotlib, default settings, one chart per figure)
import matplotlib.pyplot as plt

plt.figure()
plt.bar(stress_counts["subreddit"], stress_counts["size"])
plt.title("Top-10 Subreddits (Stress = 1)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure()
plt.bar(nonstress_counts["subreddit"], nonstress_counts["size"])
plt.title("Top-10 Subreddits (Stress = 0)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
